# Phase 5 — Dixon-Coles Model + Monte Carlo Simulation

**Goal:** Predict Portugal's 2026 WC win probability using a Poisson attack/defense model fitted by Maximum Likelihood Estimation.

### How Dixon-Coles works
1. Each team has an **attack parameter** (α) and **defense parameter** (β), fitted from historical matches
2. Expected goals for Team A vs Team B: `λ_A = exp(μ + α_A − β_B)`
3. Goals follow a **Poisson distribution** with those expected values
4. A **rho (ρ) correction** adjusts probabilities for low-scoring results (0-0, 1-0, 0-1, 1-1)
5. The full **scoreline probability matrix** (9×9) lets us derive win/draw/loss probabilities

### Training data
- International matches 2010–2025 (15,704 matches)
- **Time-weighted** — recent matches count more: `w = exp(−0.0065 × days_ago)`
- Teams with < 20 appearances get global-average parameters

In [ ]:
import sys, os
sys.path.insert(0, '..')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')   # run from project root so data/ paths resolve correctly

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

from src.dixon_coles import DixonColes, run_monte_carlo_dc, run_portugal_path_dc

CHARTS = Path('outputs/charts')
SIM    = Path('simulation')
MODELS = Path('models')
for p in [CHARTS, SIM, MODELS]: p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 130, 'font.family': 'DejaVu Sans'})
print('Ready.')

## 1. Fit the Dixon-Coles model

In [ ]:
%%time
intl = pd.read_csv('data/raw/international_results.csv')
groups_raw = pd.read_csv('data/raw/wc_2026_groups.csv')
NAME_MAP_WC = {'DR Congo': 'Democratic Republic of Congo',
               'Czech Republic': 'Czechia', 'Curacao': 'Curaçao'}
wc_teams = [NAME_MAP_WC.get(t, t) for t in groups_raw['team'].tolist()]

model = DixonColes()
model.fit(intl, target_teams=wc_teams)   # 48 WC teams only → ~98 params, fast convergence

In [ ]:
# Save fitted parameters
model.save(MODELS / 'dixon_coles_params.json')
print(f'Global mu (log avg goals): {model.mu_:.3f}  →  avg goals = {np.exp(model.mu_):.3f}')
print(f'Rho (DC correction):       {model.rho_:.4f}')
print(f'Teams fitted:              {len(model.teams_)}')

## 2. Attack / Defense parameters for WC 2026 teams

In [ ]:
groups = pd.read_csv('data/raw/wc_2026_groups.csv')
NAME_MAP = {'DR Congo': 'Democratic Republic of Congo', 'Czech Republic': 'Czechia', 'Curacao': 'Curaçao'}
groups['team'] = groups['team'].map(lambda x: NAME_MAP.get(x, x))

wc_teams = groups['team'].tolist()
params_df = model.team_params(wc_teams)
params_df = params_df.merge(groups[['team','group']], on='team', how='left')

print('Top 15 WC teams by attack parameter:')
print(params_df.head(15).to_string(index=False))

In [ ]:
# Group K
group_k = params_df[params_df['group'] == 'K']
print('Group K — Portugal\'s group:')
print(group_k.to_string(index=False))

print()
# Portugal vs Colombia
lam, mu = model.expected_goals('Portugal', 'Colombia')
pw, pd_, pl = model.match_probs('Portugal', 'Colombia')
print(f'Portugal vs Colombia:')
print(f'  Expected goals: Portugal {lam:.2f} — Colombia {mu:.2f}')
print(f'  Win={pw:.1%}  Draw={pd_:.1%}  Loss={pl:.1%}')

## 3. Portugal scoreline probability matrix

In [ ]:
# Scoreline matrix: Portugal vs Colombia (key group match)
M = model.scoreline_matrix('Portugal', 'Colombia', max_goals=5)
labels = [str(i) for i in range(6)]

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    M * 100, annot=True, fmt='.1f', cmap='YlOrRd',
    xticklabels=labels, yticklabels=labels, ax=ax,
    cbar_kws={'label': 'Probability (%)'}
)
ax.set_xlabel('Colombia goals')
ax.set_ylabel('Portugal goals')
ax.set_title('Scoreline probability matrix — Portugal vs Colombia (Dixon-Coles)')
plt.tight_layout()
plt.savefig(CHARTS / '11_scoreline_port_col.png')
plt.show()
print('Saved: 11_scoreline_port_col.png')

In [ ]:
# Attack vs Defense scatter for all 48 WC teams
fig, ax = plt.subplots(figsize=(10, 7))

highlights = {'Portugal': 'green', 'Spain': 'red', 'Argentina': 'blue',
              'France': 'purple', 'England': 'navy', 'Colombia': 'orange'}

for _, row in params_df.iterrows():
    color = highlights.get(row['team'], 'lightgray')
    alpha = 0.9 if row['team'] in highlights else 0.5
    ax.scatter(row['defense'], row['attack'], color=color, alpha=alpha, s=60)
    if row['team'] in highlights:
        ax.annotate(row['team'], (row['defense'], row['attack']),
                    textcoords='offset points', xytext=(5, 3), fontsize=8)

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Defense parameter β  (higher = weaker defense)')
ax.set_ylabel('Attack parameter α  (higher = stronger attack)')
ax.set_title('Dixon-Coles attack vs defense — 2026 WC teams')
plt.tight_layout()
plt.savefig(CHARTS / '12_attack_defense_scatter.png')
plt.show()
print('Saved: 12_attack_defense_scatter.png')

## 4. Run 100,000 Monte Carlo simulations

In [ ]:
%%time
N_SIMS = 100_000
results_dc = run_monte_carlo_dc(model, N_SIMS, seed=42)
results_dc.to_csv(SIM / 'dixon_coles_simulation_results.csv', index=False)
print(f'Simulations complete. Unique champions: {len(results_dc)}')
print()
print('Top 15 champions (DC model):')
print(results_dc.head(15).to_string(index=False))

In [ ]:
port_row = results_dc[results_dc['winner'] == 'Portugal']
port_prob_dc = port_row['probability'].values[0] if len(port_row) else 0.0
print(f'=== Portugal WC 2026 win probability (Dixon-Coles): {port_prob_dc:.2%} ===')

## 5. Portugal path analysis

In [ ]:
%%time
path_dc = run_portugal_path_dc(model, N_SIMS, seed=42)
path_df = pd.DataFrame([
    {'stage': s, 'count': c, 'probability': c / N_SIMS}
    for s, c in path_dc.items()
])
print('Portugal stage-by-stage (Dixon-Coles):')
print(path_df.to_string(index=False))

## 6. Charts

In [ ]:
# Chart — Top 10 champion probabilities (DC model)
top10_dc = results_dc.head(10).copy()
colors = ['#006600' if t == 'Portugal' else '#d62728' for t in top10_dc['winner']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top10_dc['winner'][::-1], top10_dc['probability'][::-1],
        color=colors[::-1])
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('Win probability')
ax.set_title('2026 FIFA WC — Championship probability (Dixon-Coles, 100k sims)')
for i, (_, row) in enumerate(top10_dc[::-1].iterrows()):
    ax.text(row['probability'] + 0.002, i, f"{row['probability']:.1%}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig(CHARTS / '13_dc_champion_probs.png')
plt.show()
print('Saved: 13_dc_champion_probs.png')

In [ ]:
# Model comparison — ELO vs DC
try:
    elo_results = pd.read_csv(SIM / 'elo_simulation_results.csv')
    compare = elo_results.rename(columns={'probability': 'ELO'}).merge(
        results_dc.rename(columns={'probability': 'DC'})[['winner','DC']],
        on='winner', how='outer'
    ).fillna(0).sort_values('ELO', ascending=False).head(12)

    x = np.arange(len(compare))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width/2, compare['ELO'], width, label='ELO model', color='#4393c3')
    ax.bar(x + width/2, compare['DC'],  width, label='Dixon-Coles', color='#d6604d')
    ax.set_xticks(x)
    ax.set_xticklabels(compare['winner'], rotation=35, ha='right')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_ylabel('Win probability')
    ax.set_title('ELO vs Dixon-Coles — WC 2026 championship probability')
    ax.legend()
    plt.tight_layout()
    plt.savefig(CHARTS / '14_elo_vs_dc_comparison.png')
    plt.show()
    print('Saved: 14_elo_vs_dc_comparison.png')
except FileNotFoundError:
    print('ELO results not found — run Phase 4 notebook first')

## 7. Summary

| Metric | ELO Model | Dixon-Coles |
|---|---|---|
| Portugal win probability | 3.8% | see above |
| Spain | 32.4% | see above |
| Model complexity | Low | High |
| Scoreline distribution | No | Yes |
| Time-weighted training | No | Yes |

**Key Dixon-Coles advantages over ELO:**
- Models **how many goals** a team scores, not just who wins
- Fitted on recent match history with **time decay** (recent = more important)
- The **rho correction** captures football-specific draw patterns
- Attack/defense parameters are **team-specific and interpretable**

**Known limitations (same as ELO):**
- Bracket structure is an approximation of the real FIFA 2026 bracket
- Neutral venue assumption (no home advantage term)

**Next:** Phase 6 — XGBoost classifier (ML approach using all engineered features)